# Data Preprocessing — EPL 2024/25
## ECS784P/U Data Analytics — Coursework 1

This notebook merges four FBref squad-level CSV files for the 2024–2025 English Premier League season into a single master dataset, which is then used by the main analysis notebook (`Data_Analytics_Coursework_1.ipynb`).

**Input files:**
- `epl_2024_2025_standard_stats.csv`
- `epl_2024_2025_shooting_stats.csv`
- `epl_2024_2025_misc_stats.csv`
- `epl_2024_2025_goalkeeping_stats.csv`

**Output:** `epl_2024_2025_master_stats.csv` — 20 teams × 47 columns

In [1]:
# Data manipulation
import pandas as pd

## 1. Load Raw Data Files

Load each of the four FBref CSV files and confirm their dimensions.

In [2]:
# Load the four FBref CSV files for the 2024-2025 EPL season.
# Each file covers a different statistical category:
#   standard    — possession, goals, assists, cards (21 columns)
#   shooting    — shots, shots on target, shot quality (13 columns)
#   misc        — fouls, offsides, crosses, tackles (15 columns)
#   goalkeeping — saves, clean sheets, goals against (21 columns)
standard   = pd.read_csv('epl_2024_2025_standard_stats.csv')
shooting   = pd.read_csv('epl_2024_2025_shooting_stats.csv')
misc       = pd.read_csv('epl_2024_2025_misc_stats.csv')
goalkeeping = pd.read_csv('epl_2024_2025_goalkeeping_stats.csv')

print("Standard:   ", standard.shape)
print("Shooting:   ", shooting.shape)
print("Misc:       ", misc.shape)
print("Goalkeeping:", goalkeeping.shape)

Standard:    (20, 21)
Shooting:    (20, 13)
Misc:        (20, 15)
Goalkeeping: (20, 21)


### 1.1 Inspect Column Names

Print column names from each file to identify overlapping columns that need to be handled during the merge.

In [3]:
# Inspect column names in each file to identify overlapping columns.
# Overlaps (e.g. '# Pl', '90s', 'CrdY') must be handled during merging
# to avoid duplicate columns in the master dataset.
print("Standard columns:   ", standard.columns.tolist())
print("\nShooting columns:   ", shooting.columns.tolist())
print("\nMisc columns:       ", misc.columns.tolist())
print("\nGoalkeeping columns:", goalkeeping.columns.tolist())

Standard columns:    ['Squad', '# Pl', 'Age', 'Poss', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK']

Shooting columns:    ['Squad', '# Pl', '90s', 'Gls', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK', 'PKatt']

Misc columns:        ['Squad', '# Pl', '90s', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'PKwon', 'PKcon', 'OG']

Goalkeeping columns: ['Squad', '# Pl', 'MP', 'Starts', 'Min', '90s', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt', 'PKA', 'PKsv', 'PKm', 'Save%.1']


## 2. Merge Datasets

Merge the four files step-by-step on `Squad`. Duplicate columns introduced by each join are tagged with a `_drop` suffix and removed immediately, keeping the master dataset clean.

In [4]:
# Merge all four files into one master DataFrame.
# Strategy: merge on 'Squad' one file at a time.
# suffixes=('', '_drop') marks duplicate columns from the right file with '_drop',
# which are then removed immediately after each merge to keep the dataset clean.

# Step 1: standard + shooting
master = standard.merge(shooting, on='Squad', suffixes=('', '_drop'))
master = master.drop(columns=[c for c in master.columns if c.endswith('_drop')])

# Step 2: add misc stats
master = master.merge(misc, on='Squad', suffixes=('', '_drop'))
master = master.drop(columns=[c for c in master.columns if c.endswith('_drop')])

# Step 3: add goalkeeping stats
master = master.merge(goalkeeping, on='Squad', suffixes=('', '_drop'))
master = master.drop(columns=[c for c in master.columns if c.endswith('_drop')])

# Drop '.1' suffixed columns introduced by standard stats (per-90 duplicates)
master = master.drop(columns=[c for c in master.columns if c.endswith('.1')])

print("Master dataset shape:", master.shape)
master.columns.tolist()

Master dataset shape: (20, 47)


['Squad',
 '# Pl',
 'Age',
 'Poss',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'G+A-PK',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 '2CrdY',
 'Fls',
 'Fld',
 'Off',
 'Crs',
 'Int',
 'TklW',
 'PKwon',
 'PKcon',
 'OG',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKA',
 'PKsv',
 'PKm']

## 3. Inspect Master Dataset

Verify the merged shape and column list before exporting.

In [5]:
# Preview the first 5 rows to confirm the merge looks correct
master.head()

,Squad,# Pl,Age,Poss,MP,Starts,Min,90s,Gls,Ast,...,Saves,Save%,W,D,L,CS,CS%,PKA,PKsv,PKm
0,Arsenal,25,25.8,56.9,38,418,"3,420",38.0,1.76,1.45,...,2.26,74.2,0.53,0.37,0.11,0.34,34.2,0.08,0.00,0.0
1,Aston Villa,28,27.0,50.5,38,418,"3,420",38.0,1.47,1.18,...,2.74,68.6,0.50,0.24,0.26,0.24,23.7,0.05,0.03,0.0
2,Bournemouth,29,25.1,48.5,38,418,"3,420",38.0,1.50,1.08,...,3.32,75.6,0.39,0.29,0.32,0.24,23.7,0.11,0.00,0.0
3,Brentford,28,25.8,47.9,38,418,"3,420",38.0,1.71,1.16,...,4.03,72.9,0.42,0.21,0.37,0.21,21.1,0.03,0.00,0.0
4,Brighton,32,24.8,52.3,38,418,"3,420",38.0,1.68,1.08,...,2.37,66.0,0.42,0.34,0.24,0.21,21.1,0.24,0.00,0.0


## 4. Export Master Dataset

In [6]:
# Export the master dataset to CSV for use in the main analysis notebook
master.to_csv('epl_2024_2025_master_stats.csv', index=False)
print("Exported: epl_2024_2025_master_stats.csv", master.shape)

Exported: epl_2024_2025_master_stats.csv (20, 47)
